# 01 · Setup & Data
GPU/bf16 check, clone+install, Kaggle download, data inspection, cache build, stratified 70/15/15 split, and EDA visuals (class distribution, sample grid, augmentation preview).

In [ ]:
# === Preamble 1/2: environment & GPU report ===
# This is a REMOTE Colab kernel — it cannot see your local files.
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('bfloat16 supported:', torch.cuda.is_bf16_supported())
    else:
        print('No GPU — Runtime > Change runtime type > A100 (or L4).')
except ImportError:
    print('torch installs in the next cell.')

In [ ]:
# === Preamble 2/2: clone-or-pull + install (+ optional autoreload) ===
import os, subprocess, sys

REPO_URL = "https://github.com/Kidhurshan/plant-leaf-classifier.git"  # <-- EDIT to your repo
REPO_DIR = "/content/plant-leaf-classifier"
# Private repo? use https://<TOKEN>@github.com/Kidhurshan/plant-leaf-classifier.git

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'requirements.txt'], check=True)

# Hot-reload src/ after a `git pull` (optional convenience; never fatal).
try:
    from IPython import get_ipython
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload enabled.')
except Exception as _e:
    print('autoreload not enabled (non-fatal):', repr(_e))

from src.utils import sync_repo, gpu_report
sync_repo()   # git pull + print the commit hash these results are traceable to
gpu_report()

## 1 · Configuration & seed

In [ ]:
from src.config import load_config
from src.utils import set_seed, detect_amp
import torch

cfg = load_config('configs/default.yaml')
cfg.paths.ensure_dirs()
set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = detect_amp(device)
print('device:', device, '| AMP:', amp.dtype_name,
      '| classes:', cfg.data.num_classes)

## 2 · (Optional) mount Drive for durable checkpoints
Uncomment to survive dropped sessions.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# cfg.paths.checkpoint_dir = '/content/drive/MyDrive/task4_ckpts'
# import os; os.makedirs(cfg.paths.checkpoint_dir, exist_ok=True)
print('checkpoint_dir:', cfg.paths.checkpoint_dir)

## 3 · Kaggle authentication
Credentials live on the **runtime**, never in the repo. Two ways:
1. **Colab Secrets** 🔑 (left sidebar): add `KAGGLE_USERNAME` and `KAGGLE_KEY`, toggle notebook access on — then this cell finds them automatically.
2. **Prompt below**: if no secret/env var is set, paste the key at the hidden prompt. It is NOT stored in the notebook.

In [ ]:
import os, getpass
from src.data import ensure_kaggle_credentials

# If a Colab Secret / env var / ~/.kaggle/kaggle.json already exists,
# this prompt is skipped. Username is not secret; the key is entered hidden.
if not (os.environ.get('KAGGLE_KEY') or ensure_kaggle_credentials()):
    os.environ['KAGGLE_USERNAME'] = os.environ.get('KAGGLE_USERNAME') or 'Kidhurshan'
    os.environ['KAGGLE_KEY'] = getpass.getpass('Paste your Kaggle API key (hidden): ')
print('Kaggle credentials ready:', ensure_kaggle_credentials())

## 4 · Download & inspect (adapts to the real layout)
Merges nested healthy/diseased folders into one species label; prints the tree, counts, dimensions/formats, corrupt files, and the final class map; asserts 8 classes.

In [ ]:
from src.data import download_dataset, inspect_dataset
download_dataset(cfg.data.kaggle_slug, cfg.paths.data_dir)
disc = inspect_dataset(cfg.paths.data_dir,
                       expected_classes=cfg.data.num_classes,
                       expected_total=cfg.data.expected_total)

## 5 · Cache the dataset + write the stratified split
Decoded once to a uint8 tensor; split written **once** so all three models see identical data. A small smoke cache is also built so `--smoke` runs work.

In [ ]:
from src.data import (build_cache, cache_path, make_stratified_splits,
                      write_splits_csv, splits_csv_path,
                      stratified_subset_indices)
fracs = (cfg.data.split.train, cfg.data.split.val, cfg.data.split.test)

# --- full ---
build_cache(disc, cfg.data.cache_size, cache_path(cfg.paths.cache_dir))
split = make_stratified_splits(disc.labels, fracs, cfg.seed)
write_splits_csv([str(p) for p in disc.paths], disc.labels,
                 disc.class_names, split,
                 splits_csv_path(cfg.paths.metrics_dir))

# --- smoke subset ---
sub = stratified_subset_indices(disc.labels, cfg.smoke.n_images, cfg.seed)
build_cache(disc, cfg.data.cache_size,
            cache_path(cfg.paths.cache_dir, smoke=True), subset_idx=sub)
ssplit = make_stratified_splits(disc.labels[sub], fracs, cfg.seed)
write_splits_csv([str(disc.paths[i]) for i in sub], disc.labels[sub],
                 disc.class_names, ssplit,
                 splits_csv_path(cfg.paths.metrics_dir, smoke=True))
print('Caches + splits written.')

## 6 · Visual 1 — class distribution

In [ ]:
from src import viz
from src.data import load_cache, cache_path
viz.plot_class_distribution(
    disc.class_counts,
    out_path=f'{cfg.paths.figures_dir}/class_distribution.png');

## 7 · Visual 2 — sample image per species

In [ ]:
cache = load_cache(cache_path(cfg.paths.cache_dir))
viz.plot_sample_grid(cache['images'], cache['labels'].numpy(),
                     cache['class_names'],
                     out_path=f'{cfg.paths.figures_dir}/sample_grid.png');

## 8 · Visual 3 — augmentation preview (same image)

In [ ]:
from src.augment import GPUAugment, denormalize
aug = GPUAugment(cfg.augment, cfg.data.img_size, device, training=True)
one = cache['images'][0:1].to(device)
before = viz.to_hwc_uint8(cache['images'][0])
afters = [denormalize(aug(one))[0] for _ in range(5)]
viz.plot_augmentation_preview(
    before, afters,
    out_path=f'{cfg.paths.figures_dir}/augmentation_preview.png');

## Outputs

In [ ]:
print('figures ->', cfg.paths.figures_dir)
print('cache   ->', cache_path(cfg.paths.cache_dir))
print('splits  ->', splits_csv_path(cfg.paths.metrics_dir))

---
### ⚠️ When finished: disconnect and DELETE the runtime
`Runtime > Disconnect and delete runtime`. Colab compute units are consumed the whole time a runtime is connected.